# 5. Squat Feature Extraction (Google Colab)

This notebook starts after YOLO pose extraction is complete.

Input:
- `pose_feature_index_squat.csv`
- YOLO pose feature `.npy` files with shape `[T, 51]`

Output:
- processed squat feature `.npy` files
- a feature index CSV
- a processing summary CSV

## 1. Paths and Runtime

Use this notebook after the squat-only YOLO extraction notebook has already produced pose features.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/CV_Image_pose_detection')
ANNOTATION_DIR = REPO_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned'
POSE_INDEX_CSV = ANNOTATION_DIR / 'pose_feature_index_squat.csv'
POSE_FEATURE_DIR = ANNOTATION_DIR / 'pose_features'

OUTPUT_DIR = ANNOTATION_DIR / 'squat_features'
OUTPUT_INDEX_CSV = ANNOTATION_DIR / 'squat_feature_index.csv'
OUTPUT_SUMMARY_CSV = ANNOTATION_DIR / 'squat_feature_summary.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('POSE_INDEX_CSV =', POSE_INDEX_CSV)
print('POSE_FEATURE_DIR =', POSE_FEATURE_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 2. Imports

In [ ]:
import csv
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. Load Squat Pose Index

In [ ]:
pose_index = pd.read_csv(POSE_INDEX_CSV)
pose_index.head()

In [ ]:
print('rows =', len(pose_index))
print('missing pose files =', int((~pose_index['feature_path'].map(lambda p: Path(p).exists())).sum()))

## 4. COCO Keypoint Layout

YOLO pose uses 17 COCO keypoints.

In [ ]:
KEYPOINT_NAMES = [
    'nose',
    'left_eye', 'right_eye',
    'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist',
    'left_hip', 'right_hip',
    'left_knee', 'right_knee',
    'left_ankle', 'right_ankle',
]

KPT = {name: idx for idx, name in enumerate(KEYPOINT_NAMES)}
KPT

## 5. Helper Functions

In [ ]:
CONF_THRESHOLD = 0.25
EMA_ALPHA = 0.2
MIN_SCALE = 1e-6


def load_pose_array(path: Path) -> np.ndarray:
    arr = np.load(path)
    if arr.ndim != 2 or arr.shape[1] != 51:
        raise ValueError(f'Expected [T, 51], got {arr.shape} for {path}')
    return arr.astype(np.float32)


def reshape_pose(arr: np.ndarray) -> np.ndarray:
    return arr.reshape(arr.shape[0], 17, 3)


def forward_fill_nan(xy: np.ndarray) -> np.ndarray:
    out = xy.copy()
    for t in range(1, out.shape[0]):
        missing = np.isnan(out[t])
        out[t][missing] = out[t - 1][missing]
    return out


def backward_fill_nan(xy: np.ndarray) -> np.ndarray:
    out = xy.copy()
    for t in range(out.shape[0] - 2, -1, -1):
        missing = np.isnan(out[t])
        out[t][missing] = out[t + 1][missing]
    return out


def ema_smooth(xy: np.ndarray, alpha: float = EMA_ALPHA) -> np.ndarray:
    out = xy.copy()
    for t in range(1, out.shape[0]):
        out[t] = alpha * out[t] + (1.0 - alpha) * out[t - 1]
    return out


def preprocess_pose(pose: np.ndarray, conf_threshold: float = CONF_THRESHOLD, alpha: float = EMA_ALPHA):
    xy = pose[:, :, :2].copy()
    conf = pose[:, :, 2].copy()

    xy[conf < conf_threshold] = np.nan
    valid_mask = conf >= conf_threshold

    xy = forward_fill_nan(xy)
    xy = backward_fill_nan(xy)
    xy = np.nan_to_num(xy, nan=0.0)
    xy = ema_smooth(xy, alpha=alpha)
    return xy, conf, valid_mask


def midpoint(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return (a + b) / 2.0


def vector_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    ba = a - b
    bc = c - b
    dot = np.sum(ba * bc, axis=-1)
    norm_ba = np.linalg.norm(ba, axis=-1)
    norm_bc = np.linalg.norm(bc, axis=-1)
    denom = np.clip(norm_ba * norm_bc, 1e-6, None)
    cosine = np.clip(dot / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cosine))


def normalize_pose(xy: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    left_hip = xy[:, KPT['left_hip']]
    right_hip = xy[:, KPT['right_hip']]
    left_shoulder = xy[:, KPT['left_shoulder']]
    right_shoulder = xy[:, KPT['right_shoulder']]

    hip_center = midpoint(left_hip, right_hip)
    shoulder_center = midpoint(left_shoulder, right_shoulder)
    scale = np.linalg.norm(shoulder_center - hip_center, axis=-1)
    scale = np.clip(scale, MIN_SCALE, None)

    normalized = (xy - hip_center[:, None, :]) / scale[:, None, None]
    return normalized, hip_center, scale


def build_frame_features(normalized_xy: np.ndarray, conf: np.ndarray, valid_mask: np.ndarray) -> pd.DataFrame:
    left_knee_angle = vector_angle(
        normalized_xy[:, KPT['left_hip']],
        normalized_xy[:, KPT['left_knee']],
        normalized_xy[:, KPT['left_ankle']],
    )
    right_knee_angle = vector_angle(
        normalized_xy[:, KPT['right_hip']],
        normalized_xy[:, KPT['right_knee']],
        normalized_xy[:, KPT['right_ankle']],
    )

    left_hip_angle = vector_angle(
        normalized_xy[:, KPT['left_shoulder']],
        normalized_xy[:, KPT['left_hip']],
        normalized_xy[:, KPT['left_knee']],
    )
    right_hip_angle = vector_angle(
        normalized_xy[:, KPT['right_shoulder']],
        normalized_xy[:, KPT['right_hip']],
        normalized_xy[:, KPT['right_knee']],
    )

    hip_center_y = midpoint(normalized_xy[:, KPT['left_hip']], normalized_xy[:, KPT['right_hip']])[:, 1]
    knee_center_y = midpoint(normalized_xy[:, KPT['left_knee']], normalized_xy[:, KPT['right_knee']])[:, 1]
    ankle_center_y = midpoint(normalized_xy[:, KPT['left_ankle']], normalized_xy[:, KPT['right_ankle']])[:, 1]

    avg_knee_angle = (left_knee_angle + right_knee_angle) / 2.0
    avg_hip_angle = (left_hip_angle + right_hip_angle) / 2.0
    knee_flex = 180.0 - avg_knee_angle
    hip_drop = knee_center_y - hip_center_y
    leg_extension = ankle_center_y - hip_center_y
    hip_velocity = np.gradient(hip_center_y)
    frame_valid = (
        valid_mask[:, KPT['left_hip']] & valid_mask[:, KPT['right_hip']] &
        valid_mask[:, KPT['left_knee']] & valid_mask[:, KPT['right_knee']] &
        valid_mask[:, KPT['left_ankle']] & valid_mask[:, KPT['right_ankle']]
    ).astype(np.int32)

    return pd.DataFrame({
        'frame_idx': np.arange(len(normalized_xy), dtype=np.int32),
        'left_knee_angle': left_knee_angle,
        'right_knee_angle': right_knee_angle,
        'avg_knee_angle': avg_knee_angle,
        'knee_flex': knee_flex,
        'left_hip_angle': left_hip_angle,
        'right_hip_angle': right_hip_angle,
        'avg_hip_angle': avg_hip_angle,
        'hip_center_y': hip_center_y,
        'knee_center_y': knee_center_y,
        'ankle_center_y': ankle_center_y,
        'hip_drop': hip_drop,
        'leg_extension': leg_extension,
        'hip_velocity': hip_velocity,
        'frame_valid': frame_valid,
        'mean_conf': conf.mean(axis=1),
    })


## 6. Inspect One Pose File

In [ ]:
sample_row = pose_index.iloc[0]
sample_pose_path = Path(sample_row['feature_path'])
sample_arr = load_pose_array(sample_pose_path)
sample_pose = reshape_pose(sample_arr)
sample_xy, sample_conf, sample_valid = preprocess_pose(sample_pose)
sample_norm_xy, sample_hip_center, sample_scale = normalize_pose(sample_xy)
sample_features = build_frame_features(sample_norm_xy, sample_conf, sample_valid)

print('video =', sample_row['name'])
print('raw pose shape =', sample_arr.shape)
print('processed feature shape =', sample_features.shape)
sample_features.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(sample_features['avg_knee_angle'], label='avg_knee_angle')
axes[0].plot(sample_features['avg_hip_angle'], label='avg_hip_angle')
axes[0].legend()
axes[0].set_ylabel('angle')

axes[1].plot(sample_features['knee_flex'], label='knee_flex')
axes[1].plot(sample_features['hip_drop'], label='hip_drop')
axes[1].legend()
axes[1].set_ylabel('movement signal')

axes[2].plot(sample_features['mean_conf'], label='mean_conf')
axes[2].plot(sample_features['frame_valid'], label='frame_valid')
axes[2].legend()
axes[2].set_ylabel('quality')
axes[2].set_xlabel('frame')

plt.tight_layout()
plt.show()

## 7. Batch Feature Extraction

In [ ]:
@dataclass
class FeatureRunResult:
    name: str
    pose_path: str
    feature_path: str
    status: str
    frames_total: int
    frames_valid: int
    mean_conf: float
    message: str


def process_one_pose_file(row: pd.Series) -> tuple[pd.DataFrame, FeatureRunResult]:
    name = row['name']
    pose_path = Path(row['feature_path'])
    output_path = OUTPUT_DIR / f"{pose_path.stem}_squat_features.npy"

    try:
        arr = load_pose_array(pose_path)
        pose = reshape_pose(arr)
        xy, conf, valid = preprocess_pose(pose)
        normalized_xy, hip_center, scale = normalize_pose(xy)
        features = build_frame_features(normalized_xy, conf, valid)
        np.save(output_path, features.to_numpy(dtype=np.float32))
        result = FeatureRunResult(
            name=name,
            pose_path=str(pose_path),
            feature_path=str(output_path),
            status='ok',
            frames_total=int(len(features)),
            frames_valid=int(features['frame_valid'].sum()),
            mean_conf=float(features['mean_conf'].mean()),
            message='',
        )
        return features, result
    except Exception as exc:
        result = FeatureRunResult(
            name=name,
            pose_path=str(pose_path),
            feature_path=str(output_path),
            status='failed',
            frames_total=0,
            frames_valid=0,
            mean_conf=0.0,
            message=str(exc),
        )
        return pd.DataFrame(), result

In [ ]:
results = []
feature_index_rows = []

for i, row in pose_index.iterrows():
    features, result = process_one_pose_file(row)
    results.append(result.__dict__)
    if result.status == 'ok':
        feature_index_rows.append({
            'name': result.name,
            'pose_path': result.pose_path,
            'feature_path': result.feature_path,
            'type': row.get('type', 'squat'),
            'split': row.get('split', ''),
            'count': row.get('count', ''),
        })
    if (i + 1) % 25 == 0 or (i + 1) == len(pose_index):
        print(f'[{i + 1}/{len(pose_index)}] processed')

In [ ]:
summary_df = pd.DataFrame(results)
feature_index_df = pd.DataFrame(feature_index_rows)

summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
feature_index_df.to_csv(OUTPUT_INDEX_CSV, index=False)

print('saved summary =', OUTPUT_SUMMARY_CSV)
print('saved feature index =', OUTPUT_INDEX_CSV)
summary_df.head()

## 8. Run Summary

In [ ]:
print(summary_df['status'].value_counts(dropna=False))
print('\nmean valid frames =', summary_df.loc[summary_df['status'] == 'ok', 'frames_valid'].mean())
print('mean confidence =', summary_df.loc[summary_df['status'] == 'ok', 'mean_conf'].mean())

In [ ]:
summary_df.sort_values(['status', 'frames_valid', 'mean_conf']).head(10)

## 9. Output Contract for the Next Step

The next notebook can now consume:

- `squat_feature_index.csv`
- `squat_features/*.npy`

That next stage should focus on:
- choosing the best rep-count signal
- defining state transitions
- building a squat FSM counter